# Session 3 Homework · Solutions (teacher copy)

Complete solutions with teaching notes. All code runs top-to-bottom.

**Grading toward the success criterion** (concrete artifact — named metrics begin Session 8):

- all five exercises attempted
- Ex 2: `select_dtypes(include="object")` returns an empty list — no text columns remain
- Ex 3: both a training and a test score are printed
- Ex 4: a predicted price is printed next to the actual price
- Ex 5: ≥ 3 sentences using *memorizing* / *learning* correctly, plus a sentence naming a scaling-sensitive model

Accept reasonable variants: `drop_first=True` in `get_dummies`, a different `random_state`, or a manual encoding that produces 0/1 columns. The expected test R² for this seeded dataset is high (roughly 0.9+) because price is built as a mostly-linear function of the features.

## Exercise 1 · Load and find the text column

In [ ]:
import pandas as pd
import numpy as np

# A reproducible dataset of laptop listings (fixed seed -> same data every run).
rng = np.random.default_rng(3)
n = 60
brands = np.array(["Acer", "Dell", "Apple", "Lenovo"])

brand = rng.choice(brands, size=n)
ram_gb = rng.choice([8, 16, 32], size=n)
ssd_gb = rng.choice([256, 512, 1024], size=n)
screen_inch = np.round(rng.uniform(13, 17, size=n), 1)
weight_kg = np.round(rng.uniform(1.0, 2.4, size=n), 2)

brand_premium = np.select(
    [brand == "Apple", brand == "Dell"],
    [25.0, 5.0],
    default=0.0,
)
price_thousands = np.round(
    15 + ram_gb * 0.9 + ssd_gb * 0.03 + brand_premium
    + screen_inch * 0.5 - weight_kg * 4
    + rng.normal(0, 3, size=n),
    1,
)

laptops = pd.DataFrame({
    "brand": brand,
    "ram_gb": ram_gb,
    "ssd_gb": ssd_gb,
    "screen_inch": screen_inch,
    "weight_kg": weight_kg,
    "price_thousands": price_thousands,
})
laptops.head()

In [ ]:
print("shape:", laptops.shape)
laptops.info()

**Expected answer:** `brand` is the text column (`object` dtype). Everything else is numeric. (`price_thousands` is the label, not a feature.)

## Exercise 2 · One-hot encode the text column

In [ ]:
laptops_encoded = pd.get_dummies(laptops, columns=["brand"], dtype=int)
laptops_encoded.head()

In [ ]:
text_columns_left = list(laptops_encoded.select_dtypes(include="object").columns)
print("text columns remaining:", text_columns_left)

**Expected answer:** the encoding creates `brand_Acer`, `brand_Apple`, `brand_Dell`, `brand_Lenovo`, and `select_dtypes(include="object")` is now empty. Numbering brands 0–3 would invent a false ranking and equal spacing between brands; one-hot keeps them as independent yes/no facts.

*Teaching note:* if a student used `drop_first=True`, they'll have three brand columns, not four — accept it and explain it avoids a redundant column. Don't require it.

## Exercise 3 · Split, then run the three moves

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

y = laptops_encoded["price_thousands"]
X = laptops_encoded.drop(columns=["price_thousands"])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

model = LinearRegression()
model.fit(X_train, y_train)

print("training score:", round(model.score(X_train, y_train), 3))
print("test score:    ", round(model.score(X_test, y_test), 3))

**Expected answer:** both scores print and are high (≈ 0.9+). The training score is usually a little higher than the test score, because the model has *seen* the training rows — that small gap is normal and is exactly the memorizing-vs-learning check from class. A student who sees a much lower test score should check they didn't accidentally leave `brand` unencoded.

## Exercise 4 · Predict one held-out laptop

In [ ]:
first_prediction = model.predict(X_test.iloc[[0]])[0]
first_actual = y_test.iloc[0]

print(f"predicted: {first_prediction:.1f} thousand")
print(f"actual:    {first_actual:.1f} thousand")
print(f"off by:    {abs(first_prediction - first_actual):.1f} thousand")

**Expected answer:** the prediction should land within a few thousand of the actual price. A score across many held-out laptops is more trustworthy than a single prediction — one lucky (or unlucky) row tells you little; the test score summarises performance over the whole hidden set.

*Teaching note:* this previews the Session 8 message that we judge models on aggregate metrics, not anecdotes.

## Exercise 5 · Why hide data? (open-ended)

**Model answer (accept any well-reasoned variant):**

We scored the model on laptops it never saw during training because a high score on the training data could just mean it *memorized* those exact rows, the way the Session 1 model memorized an answer key and scored a perfect 100% while being useless on anything new. Scoring on held-out laptops checks whether the model actually *learned* the pattern connecting specs to price — a pattern that should work on laptops it has never encountered. If the model had only memorized, its test score would collapse, exposing it.

*Scaling sentence (accept any of these):* a distance-based model such as **KNN** (Module 3) needs scaled features, because otherwise a big-numbered feature like SSD size would dominate the distance calculation and drown out smaller-numbered features.

*Grading:* require at least three sentences and correct use of *memorizing* / *learning*. The scaling sentence should name a model or situation where scale affects the result (KNN, k-means, or any "distance"/"magnitude"-based method).